# UCSF-BMSR MET 학습 노트북
**모델**: SegResNet (init_filters=32, 18.8M params)  
**데이터**: UCSF-BMSR (259 train / 65 val)  
**GPU 권장**: A100 (Colab pay-as-you-go)

### 주의사항
- **A100 선택**: 런타임 → 런타임 유형 변경 → A100 GPU
- T4 사용 시 `init_filters=16`으로 변경 필요
- cache_dir은 로컬 `/content/`에 저장 (Drive 용량 문제 방지)

In [ ]:
# ════════════════════════════════════════════════════════
# Cell 1: 환경 설정 및 라이브러리
# ════════════════════════════════════════════════════════
!pip install monai[all] nibabel einops -q

import os, random, shutil
import numpy as np
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler
from tqdm.auto import tqdm

from monai.transforms import (
    LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    NormalizeIntensityd, RandFlipd, RandRotate90d,
    RandScaleIntensityd, RandShiftIntensityd,
    RandSpatialCropd, Lambdad, Compose
)
from monai.data import PersistentDataset, DataLoader, decollate_batch
from monai.networks.nets import SegResNet
from monai.losses import DiceFocalLoss
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete
from monai.inferers import sliding_window_inference

# ── 디바이스 ────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# ── 공통 함수 ──────────────────────────────────────────
def ensure_4ch(x: torch.Tensor) -> torch.Tensor:
    if x.ndim == 5 and x.shape[-1] == 1:
        return x.squeeze(-1)
    return x

def brats_label_convert(label: torch.Tensor) -> torch.Tensor:
    out    = torch.zeros((3,) + label.shape[1:], dtype=torch.float32)
    out[0] = ((label[0]==1)|(label[0]==2)|(label[0]==3)).float()  # WT
    out[1] = ((label[0]==1)|(label[0]==3)).float()                 # TC
    out[2] = (label[0]==3).float()                                 # ET
    return out

# ── 전처리 Transform ────────────────────────────────────
ROI_SIZE = (128, 128, 128)

train_transform = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Lambdad(keys="image", func=ensure_4ch),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(keys=["image", "label"],
             pixdim=(1.0, 1.0, 1.0),
             mode=("bilinear", "nearest")),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    Lambdad(keys="label", func=brats_label_convert),
    RandSpatialCropd(
        keys=["image", "label"],
        roi_size=ROI_SIZE,
        random_size=False,
    ),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
    RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
    RandScaleIntensityd(keys="image", factors=0.1, prob=0.5),
    RandShiftIntensityd(keys="image", offsets=0.1, prob=0.5),
])

val_transform = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Lambdad(keys="image", func=ensure_4ch),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(keys=["image", "label"],
             pixdim=(1.0, 1.0, 1.0),
             mode=("bilinear", "nearest")),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    Lambdad(keys="label", func=brats_label_convert),
])

print("✅ Cell 1 완료")

In [ ]:
# ════════════════════════════════════════════════════════
# Cell 2: Google Drive 마운트 + 데이터 압축 해제
# ════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

# ⚠️ 폴더명에 공백 있음 (실제 Drive 폴더: " brain-tumor-diagnosis")
DRIVE_DIR   = "/content/drive/MyDrive/ brain-tumor-diagnosis"
ZIP_PATH    = f"{DRIVE_DIR}/UCSF_BrainMetastases_v1.3.zip"
EXTRACT_DIR = "/content/UCSF-BMSR"
DATA_ROOT   = f"{EXTRACT_DIR}/UCSF_BrainMetastases_TRAIN"

# ── 체크포인트 경로는 Drive에 저장 (세션 재시작 후에도 유지) ──
SAVE_DIR        = DRIVE_DIR
CHECKPOINT_PATH = f"{SAVE_DIR}/checkpoint_ucsf.pth"
BEST_MODEL_PATH = f"{SAVE_DIR}/best_ucsf_segresnet.pth"

print(f"Drive 폴더 확인: {os.path.exists(DRIVE_DIR)}")
print(f"ZIP 존재: {os.path.exists(ZIP_PATH)}")
print(f"기존 체크포인트 존재: {os.path.exists(CHECKPOINT_PATH)}")

# ── 압축 해제 (이미 완료 시 스킵) ──────────────────────
if os.path.exists(DATA_ROOT) and len(os.listdir(DATA_ROOT)) > 400:
    print("✅ 이미 압축 해제됨, 스킵")
else:
    print("압축 해제 중... (약 5~10분 소요)")
    import zipfile
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print("✅ 압축 해제 완료")

In [ ]:
# ════════════════════════════════════════════════════════
# Cell 3: 데이터 목록 구성
# ════════════════════════════════════════════════════════
cases            = sorted(os.listdir(DATA_ROOT))
data_list        = []
missing_or_empty = []

for case in cases:
    case_dir = os.path.join(DATA_ROOT, case)
    if not os.path.isdir(case_dir):
        continue
    files = {
        "t1pre"  : os.path.join(case_dir, f"{case}_T1pre.nii.gz"),
        "t1post" : os.path.join(case_dir, f"{case}_T1post.nii.gz"),
        "flair"  : os.path.join(case_dir, f"{case}_FLAIR.nii.gz"),
        "t2synth": os.path.join(case_dir, f"{case}_T2Synth.nii.gz"),
        "seg"    : os.path.join(case_dir, f"{case}_BraTS-seg.nii.gz"),
    }
    is_valid = all(
        os.path.exists(p) and os.path.getsize(p) > 0
        for p in files.values()
    )
    if is_valid:
        data_list.append({
            "image": [files["t1pre"], files["t1post"],
                      files["flair"], files["t2synth"]],
            "label": files["seg"],
        })
    else:
        missing_or_empty.append(case)

random.seed(42)
random.shuffle(data_list)

split       = int(len(data_list) * 0.8)
train_files = data_list[:split]
val_files   = data_list[split:]
print(f"✅ 유효 케이스: {len(data_list)} | Train: {len(train_files)} | Val: {len(val_files)}")
print(f"❌ 제외 케이스: {len(missing_or_empty)}")

In [ ]:
# ════════════════════════════════════════════════════════
# Cell 4: Dataset & DataLoader
# ✅ 핵심 수정: cache_dir = 로컬 /content/ (Drive가 아님)
#    → Drive 용량 부족으로 인한 OSError [Errno 28] 방지
# ════════════════════════════════════════════════════════

# ⚠️ 로컬 캐시: 세션 재시작 시 재생성 필요 (첫 epoch만 느림)
# ⚠️ Drive 캐시는 사용 금지 (용량 부족 → 학습 중단 원인)
CACHE_DIR = "/content/persistent_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"캐시 경로: {CACHE_DIR} (로컬 — Drive 아님)")

train_ds = PersistentDataset(
    data=train_files,
    transform=train_transform,
    cache_dir=CACHE_DIR
)
val_ds = PersistentDataset(
    data=val_files,
    transform=val_transform,
    cache_dir=CACHE_DIR
)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,
                          num_workers=2, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False,
                          num_workers=2, pin_memory=False)

print("배치 shape 확인 중...")
tb = next(iter(train_loader))
print(f"✅ Image: {tb['image'].shape} | Label: {tb['label'].shape}")
del tb
torch.cuda.empty_cache()

In [ ]:
# ════════════════════════════════════════════════════════
# Cell 5: 모델 정의
# ════════════════════════════════════════════════════════
torch.cuda.empty_cache()

# ── GPU VRAM별 init_filters 선택 ──────────────────────
# A100 (40GB) → init_filters=32  (권장, 성능 우선)
# V100 (16GB) → init_filters=32  (가능)
# T4   (16GB) → init_filters=16  (OOM 방지)
INIT_FILTERS = 32  # A100 사용 시; T4이면 16으로 변경

model = SegResNet(
    spatial_dims=3,
    init_filters=INIT_FILTERS,
    in_channels=4,
    out_channels=3,
    dropout_prob=0.2,
).to(device)

loss_function = DiceFocalLoss(to_onehot_y=False, sigmoid=True, gamma=2.0)
optimizer     = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
MAX_EPOCHS    = 50
VAL_EVERY     = 1
PATIENCE      = 15
scheduler     = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

n_params = sum(p.numel() for p in model.parameters())
print(f"✅ SegResNet (init_filters={INIT_FILTERS}): {n_params:,} 파라미터")

allocated = torch.cuda.memory_allocated() / 1024**3
reserved  = torch.cuda.memory_reserved()  / 1024**3
print(f"   GPU 사용: {allocated:.2f} GB | 예약: {reserved:.2f} GB")
print(f"   체크포인트 경로: {CHECKPOINT_PATH}")

In [ ]:
# ════════════════════════════════════════════════════════
# Cell 6: 학습 루프 (체크포인트 이어서 학습 지원)
# ════════════════════════════════════════════════════════
torch.cuda.empty_cache()

scaler      = GradScaler()
dice_metric = DiceMetric(include_background=True, reduction="mean_batch")
post_trans  = AsDiscrete(threshold=0.5)

# ── 체크포인트 로드 ──────────────────────────────────────
START_EPOCH        = 0
best_metric        = -1
best_metric_epoch  = -1
no_improve_count   = 0
train_loss_history = []
val_metric_history = []

if os.path.exists(CHECKPOINT_PATH):
    print(f"체크포인트 로드 중: {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    scaler.load_state_dict(ckpt["scaler"])
    START_EPOCH        = ckpt["epoch"] + 1
    best_metric        = ckpt.get("best_metric", -1)
    best_metric_epoch  = ckpt.get("best_metric_epoch", -1)
    no_improve_count   = ckpt.get("no_improve_count", 0)
    train_loss_history = ckpt.get("train_loss_history", [])
    val_metric_history = ckpt.get("val_metric_history", [])
    print(f"✅ epoch {START_EPOCH}부터 재개 | Best Dice: {best_metric:.4f} | no_improve: {no_improve_count}/{PATIENCE}")
else:
    print("🆕 처음부터 학습 시작")

# ── 학습 루프 ────────────────────────────────────────────
epoch_pbar = tqdm(range(START_EPOCH, MAX_EPOCHS), desc="Epochs")

for epoch in epoch_pbar:
    model.train()
    epoch_loss = 0.0

    for batch_data in train_loader:
        inputs = batch_data["image"].to(device)
        labels = batch_data["label"].to(device)
        optimizer.zero_grad()
        with autocast():
            outputs = model(inputs)
            loss    = loss_function(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()

    epoch_loss /= len(train_loader)
    train_loss_history.append(epoch_loss)
    scheduler.step()
    print(f"Epoch {epoch+1:3d} | Loss: {epoch_loss:.4f}", flush=True)

    # ── 검증 ─────────────────────────────────────────────
    if (epoch + 1) % VAL_EVERY == 0:
        model.eval()
        dice_metric.reset()

        with torch.no_grad():
            for val_data in val_loader:
                val_inputs = val_data["image"].to(device)
                val_labels = val_data["label"].to(device)
                with autocast():
                    val_outputs = sliding_window_inference(
                        val_inputs, ROI_SIZE,
                        sw_batch_size=2, predictor=model, overlap=0.5
                    )
                val_outputs = [post_trans(i) for i in
                               decollate_batch(torch.sigmoid(val_outputs))]
                dice_metric(y_pred=val_outputs, y=val_labels)

        channel_dice   = dice_metric.aggregate()
        current_metric = channel_dice.mean().item()
        val_metric_history.append(current_metric)

        print(f"         Val  | WT={channel_dice[0]:.4f} TC={channel_dice[1]:.4f} "
              f"ET={channel_dice[2]:.4f} | Mean={current_metric:.4f}", flush=True)

        if current_metric > best_metric:
            best_metric       = current_metric
            best_metric_epoch = epoch + 1
            no_improve_count  = 0
            ckpt_data = {
                "epoch"             : epoch,
                "model"             : model.state_dict(),
                "optimizer"         : optimizer.state_dict(),
                "scheduler"         : scheduler.state_dict(),
                "scaler"            : scaler.state_dict(),
                "best_metric"       : best_metric,
                "best_metric_epoch" : best_metric_epoch,
                "no_improve_count"  : no_improve_count,
                "train_loss_history": train_loss_history,
                "val_metric_history": val_metric_history,
            }
            torch.save(ckpt_data, BEST_MODEL_PATH)
            torch.save(ckpt_data, CHECKPOINT_PATH)
            print(f"         💾 Best 갱신 & 체크포인트 저장 @ epoch {best_metric_epoch}, Dice={best_metric:.4f}")
        else:
            no_improve_count += 1
            # best 아니어도 매 epoch 체크포인트 저장 (세션 중단 대비)
            torch.save({
                "epoch"             : epoch,
                "model"             : model.state_dict(),
                "optimizer"         : optimizer.state_dict(),
                "scheduler"         : scheduler.state_dict(),
                "scaler"            : scaler.state_dict(),
                "best_metric"       : best_metric,
                "best_metric_epoch" : best_metric_epoch,
                "no_improve_count"  : no_improve_count,
                "train_loss_history": train_loss_history,
                "val_metric_history": val_metric_history,
            }, CHECKPOINT_PATH)

        epoch_pbar.set_postfix(
            loss=f"{epoch_loss:.4f}",
            val=f"{current_metric:.4f}",
            best=f"{best_metric:.4f}",
            no_imp=no_improve_count
        )

        if no_improve_count >= PATIENCE:
            print(f"\n⏹ Early Stopping @ epoch {epoch+1} (no_improve={no_improve_count}/{PATIENCE})")
            break

print(f"\n✅ 학습 완료 | Best Dice: {best_metric:.4f} @ epoch {best_metric_epoch}")

In [ ]:
# ════════════════════════════════════════════════════════
# Cell 7: 학습 결과 시각화
# ════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training Loss
axes[0].plot(range(1, len(train_loss_history)+1), train_loss_history, label="Train Loss")
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True)
axes[0].legend()

# Validation Dice (VAL_EVERY=1이므로 epoch 1:1 대응)
val_epochs = list(range(1, len(val_metric_history)+1))
axes[1].plot(val_epochs, val_metric_history, marker="o", label="Val Dice (Mean)")
if best_metric > 0:
    axes[1].axhline(y=best_metric, color='r', linestyle='--', alpha=0.6,
                    label=f"Best: {best_metric:.4f} @ ep{best_metric_epoch}")
axes[1].set_title("Validation Dice")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Mean Dice")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
CURVE_PATH = f"{SAVE_DIR}/ucsf_training_curves.png"
plt.savefig(CURVE_PATH, dpi=150)
plt.show()
print(f"저장: {CURVE_PATH}")